<a href="https://colab.research.google.com/github/pop123-ux/HuggingFace-Project-Learning/blob/main/course/chapter7/Training_a_Data_Science_Syntax_Auto_Completer_Model(Causal%20Language%20Model)_from_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Training a causal language model from scratch (PyTorch)

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [1]:
!pip install datasets evaluate transformers[sentencepiece]
!pip install accelerate
# To run the training on TPU, you will need to uncomment the following line:
# !pip install cloud-tpu-client==0.10 torch==1.9.0 https://storage.googleapis.com/tpu-pytorch/wheels/torch_xla-1.9-cp37-cp37m-linux_x86_64.whl
!apt install git-lfs

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git-lfs is already the newest version (3.0.2-1ubuntu0.3).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.


You will need to setup git, adapt your email and name in the following cell.

In [2]:
!git config --global user.email "alexandrupp55@gmail.com"
!git config --global user.name "Pop Alexandru"

You will also need to be logged in to the Hugging Face Hub. Execute the following and enter your credentials.

In [3]:
from huggingface_hub import notebook_login

notebook_login()

In [4]:
def any_keyword_in_string(string, keywords):
    for keyword in keywords:
        if keyword in string:
            return True
    return False

In [5]:
filters = ["pandas", "sklearn", "matplotlib", "seaborn"]
example_1 = "import numpy as np"
example_2 = "import pandas as pd"

print(
    any_keyword_in_string(example_1, filters), any_keyword_in_string(example_2, filters)
)

False True


In [6]:
from collections import defaultdict
from tqdm import tqdm
from datasets import Dataset


def filter_streaming_dataset(dataset, filters):
    filtered_dict = defaultdict(list)
    total = 0
    for sample in tqdm(iter(dataset)):
        total += 1
        if any_keyword_in_string(sample["content"], filters):
            for k, v in sample.items():
                filtered_dict[k].append(v)
    print(f"{len(filtered_dict['content'])/total:.2%} of data after filtering.")
    return Dataset.from_dict(filtered_dict)

In [7]:
from datasets import load_dataset

split = "train"  # "valid"
filters = ["pandas", "sklearn", "matplotlib", "seaborn"]

data = load_dataset(f"transformersbook/codeparrot-{split}", split=split, streaming=True)
filtered_data = filter_streaming_dataset(data, filters)

README.md:   0%|          | 0.00/583 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


Resolving data files:   0%|          | 0/183 [00:00<?, ?it/s]

974469it [06:36, 2455.24it/s]


KeyboardInterrupt: 

In [8]:
from datasets import load_dataset, DatasetDict

ds_train = load_dataset("huggingface-course/codeparrot-ds-train", split="train")
ds_valid = load_dataset("huggingface-course/codeparrot-ds-valid", split="validation")

raw_datasets = DatasetDict(
    {
        "train": ds_train,  # .shuffle().select(range(50000)),
        "valid": ds_valid,  # .shuffle().select(range(500))
    }
)

raw_datasets

DatasetDict({
    train: Dataset({
        features: ['repo_name', 'path', 'copies', 'size', 'content', 'license'],
        num_rows: 606720
    })
    valid: Dataset({
        features: ['repo_name', 'path', 'copies', 'size', 'content', 'license'],
        num_rows: 3322
    })
})

In [9]:
for key in raw_datasets["train"][0]:
    print(f"{key.upper()}: {raw_datasets['train'][0][key][:200]}")

REPO_NAME: kmike/scikit-learn
PATH: sklearn/utils/__init__.py
COPIES: 3
SIZE: 10094
CONTENT: """
The :mod:`sklearn.utils` module includes various utilites.
"""

from collections import Sequence

import numpy as np
from scipy.sparse import issparse
import warnings

from .murmurhash import murm
LICENSE: bsd-3-clause


In [10]:
from transformers import AutoTokenizer

context_length = 128
tokenizer = AutoTokenizer.from_pretrained("huggingface-course/code-search-net-tokenizer")

outputs = tokenizer(
    raw_datasets["train"][:2]["content"],
    truncation=True,
    max_length=context_length,
    return_overflowing_tokens=True,
    return_length=True,
)

print(f"Input IDs length: {len(outputs['input_ids'])}")
print(f"Input chunk lengths: {(outputs['length'])}")
print(f"Chunk mapping: {outputs['overflow_to_sample_mapping']}")

Input IDs length: 34
Input chunk lengths: [128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 117, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 128, 41]
Chunk mapping: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [11]:
def tokenize(element):
    outputs = tokenizer(
        element["content"],
        truncation=True,
        max_length=context_length,
        return_overflowing_tokens=True,
        return_length=True,
    )
    input_batch = []
    for length, input_ids in zip(outputs["length"], outputs["input_ids"]):
        if length == context_length:
            input_batch.append(input_ids)
    return {"input_ids": input_batch}


tokenized_datasets = raw_datasets.map(
    tokenize, batched=True, remove_columns=raw_datasets["train"].column_names
)
tokenized_datasets

Map:   0%|          | 0/606720 [00:00<?, ? examples/s]

Map:   0%|          | 0/3322 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids'],
        num_rows: 16702061
    })
    valid: Dataset({
        features: ['input_ids'],
        num_rows: 93164
    })
})

In [12]:
from transformers import AutoTokenizer, GPT2LMHeadModel, AutoConfig

config = AutoConfig.from_pretrained(
    "gpt2",
    vocab_size=len(tokenizer),
    n_ctx=context_length,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

In [13]:
model = GPT2LMHeadModel(config)
model_size = sum(t.numel() for t in model.parameters())
print(f"GPT-2 size: {model_size/1000**2:.1f}M parameters")

GPT-2 size: 124.2M parameters


Note that DataCollatorForLanguageModeling supports both masked language modeling (MLM) and causal language modeling (CLM). By default it prepares data for MLM, but we can switch to CLM by setting the argument mlm=False:

In [1]:
from transformers import DataCollatorForLanguageModeling

tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

NameError: name 'tokenizer' is not defined

Let's take a look at an example:

In [15]:
out = data_collator([tokenized_datasets['train'][i] for i in range(5)])
for key in out:
  print(f"{key} shape: {out[key].shape}")

input_ids shape: torch.Size([5, 128])
attention_mask shape: torch.Size([5, 128])
labels shape: torch.Size([5, 128])


We can see that the examples have been stacked and all the tensors have the same shape.

## ⚠️ Shifting the inputs and labels to align them happens inside the model, so the data collator just copies the inputs to create the labels. ##

Now we have everything to train our model; all that's left to do is to configure the training arguments and to fire up the Trained. We'll use a cosine learning rate schedule with some warmup and an effective batch size of 256 (per_device_train_batch_size * gradient_accumulation_steps).

Gradient accumulation is used when a single batch does not fit into memory, and incrementally builds up the gradient through several forward/backward passes. We'll see this in action when we will create the training loop with the Accelerate API

In [16]:
from transformers import Trainer, TrainingArguments

args = TrainingArguments(
    output_dir="codeparrot-ds",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    eval_steps=5_000,
    logging_steps=5_000,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    warmup_steps=1_000,
    lr_scheduler_type='cosine',
    learning_rate=5e-4,
    save_steps=1_000,
    fp16=True,
    push_to_hub=True,
)

trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    args=args,
    data_collator=data_collator,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['valid'],
)


Now we can just start the Trainer and wait for the training to finish. Depending on whether we run it on the full or a subset of the training this will take 20 or 2 hours, respectively.

In [17]:
trainer.train()

SyntaxError: invalid syntax (1123861628.py, line 2)

After the training completes, we can push the model and tokenizer to the Hub:

In [ ]:
trainer.push_to_hub()

## Code generation with a pipeline ##

Now is the moment of truth: let's see how well the trained model actually works! We can see in the logs that the loss went down steadily, but to put the model to the test let's take a look at how well it works on some prompts. To do that we'll wrap the model in a text generation pipeline, and we'll put it on the GPU for fast generations if there is one available:

In [ ]:
import torch
from transformers import pipeline

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
pipe = pipeline(
    'text-generation', model='pop123/codeparrot-ds', device=device
)

In [ ]:
txt = """\
import torch
import matplotlib.pyplot as plt

# create some data
w = torch.tensor([0.1, 0.5, 0.1, 0.3])
s = torch.multinomial(w, 1000, replacement=True)

# create bar plot (count frequencies and plot)
"""
print(pipe(txt, num_return_sequences=1)[0]['generated_text'])

Does it also work for a pandas operation? Let's see if we can create a DataFrame from two arrays:

In [ ]:
txt = """\
# create some data
x = np.random.randn(100)
y = np.random.randn(100)

# create dataframe from x and y
"""
print(pipe(txt, num_return_sequences=1)[0]['generated_text'])

Let's see if we can do something a bit more complex and have the model help us use the groupby operation:

In [ ]:
txt = """\
# dataframe with profession, income, name, surname, household_type
df = pd.DataFrame({'profession': x, 'income':y, 'name':z, 'surname': c, 'household_type: v})

# calculate the mean income per profession
"""
print(pipe(txt, num_return_sequences=1)[0]['generated_text'])

Not bad at all; let's now see if we can also use it for scikit-learn and set up a HistGradientBoosting model:

In [ ]:
txt = """
# import hist gradient boosting classifier from scikit-learn
from sklearn.ensemble import HistGradientBoostingClassifier

# instantiate the model as "hgb" and fit it with 500 estimators on X, y:
"""
print(pipe(txt, num_return_sequences=1)[0]['generated_text'])

Looking at these few examples, it seems that the model has learned some of the syntax of the Python data science stack (of course, we would need to evaluate it more thoroughly before deploying the model in the real world). Sometimes it requires more customization of the model training to achieve the necessary performance for a given use case, however.

For example, what if we would like it to dynamically update the batch size or have a conditional training loop that skips bad examples on the fly? One option would be to subclass the Trainer and add the necessary changes, but sometimes it's simpler to write the training loop from scratch, and that's where HuggingFace Accelerate comes in.

## Training with Accelerate ##

Since we are mainly interested in sensible autocompletion for the data science libraries, it makes sens to give more weight to training samples that make more use of those libraries. We can easily identify these examples through the use of keywords such as plt, pd, sk, fit, and predict, which are the most frequent import names for matplotlib.pyplot, pandas, and sklearn as well as the fit/predict pattern of the latter. If these are each represented as a single token, we can easily check if they occur in the input sequence. Tokens can have a whitespace prefix, so we'll also check for those versions in the tokenizer vocabulary. To verify that it works, we'll add one test token which should be split into multiple tokens:

In [ ]:
keytoken_ids = []
for keyword in [
    "plt",
    "pd",
    "sk",
    "fit",
    "predict",
    " plt",
    " pd",
    " sk",
    " fit",
    " predict",
    "testtest",
    "hello",
    "Random",
]:
  ids = tokenizer([keyword]).input_ids[0]
  if len(ids) == 1:
    keytoken_ids.append(ids[0])
  else:
    print(f"Keyword has no single token: {keyword}")

We can now write a custom loss function that takes the input sequence, the logits, and the key tokens we just selected as inputs. First we need to align the logits and inputs: the input sequence shifted by one to the right forms the labels, since the next token is the label for the current token. We can achieve this by starting the labels from the second token of the input sequence, since the model does not make a prediction for the first token anyway. Then we cut off the last logit as we don't have a label for the token that follows the full input sequence. With that we can compute the loss per sample and count the occurences of all keywords in each sample. Finally, we calculate the weighted average over all samples using the occurences as weights. Since we don't want to throw away all the samples that have no keywords, we add 1 to the weights:

In [ ]:





from torch.nn import CrossEntropyLoss
import torch

def keytoken_weighted_loss(inputs, logits, keytoken_ids, alpha=1.0):
  # Shift so that tokens < n predict n
  shift_labels = inputs[..., 1:].contiguous()
  shift_logits = logits[..., :-1, :].contiguous()
  # Calculate per-token loss
  loss_fct = CrossEntropyLoss(reduce=False)
  loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
  # Resize and average loss per sample
  loss_per_sample = loss.view(shift_logits.size(0), shift_logits.size(1)).mean(axis=1)
  # Calculate and scale weighting
  weights = torch.stack([(inputs == kt).float() for kt in keytoken_ids]).sum(
      axis=[0, 2]
  )
  weights = alpha * (1.0 + weights)
  # Calculate weighted average
  weighted_loss = (loss_per_sample * weights).mean()
  return weighted_loss

Before we can start training with this awesome new loss function, we need to prepare a few things:

- We need dataloaders to load the data in batches.
-  We need to set up weight decay parameters.
- From time to time we want to evaluate, so it makes sense to wrap the evaluation code in a function.

In [ ]:
from torch.utils.data.dataloader import DataLoader

tokenized_datasets.set_format('torch')
train_dataloader = DataLoader(tokenized_datasets['train'], batch_size=32, shuffle=True)
eval_dataloader = DataLoader(tokenized_datasets['test'], batch_size=32)

Next, we group the params so that the optimizer knows which ones will get an additional weight decay. Usually, all bias and LayerNorm weights terms are exempt from this; here's how we can do this:

In [ ]:
weight_decay=0.1

def get_grouped_params(model, no_decay=['bias', 'LayerNorm.weight']):
  params_with_wd, params_without_wd = [], []
  for n, p in model.named_parameters():
    if any(nd in n for nd in no_decay):
      params_without_wd.append(p)
    else:
      params_with_wd.append(p)
  return [
      {"params": params_with_wd, "weight_decay": weight_decay},
      {"params": params_without_wd, "weight_decay": 0.0},
  ]


Since we want to evaluate the model regularly on the validation set during training, let’s write a function for that as well. It just runs through the evaluation dataloader and gathers all the losses across processes:

In [ ]:

def evaluate():
  model.eval()
  losses = []
  for step, batch in enumerate(eval_dataloader):
    with torch.no_grad():
      outputs = model(batch['input_ids'], labels=batch['input_ids'])

    losses.append(accelerator.gather(outputs.loss))
  loss = torch.mean(torch.cat(losses))
  try:
    perplexity = torch.exp(loss)
  except OverflowError:
    perplexity = float('inf')
  return loss.item(), perplexity.item()

With the evaluate() function we can report loss and perplexity at regular intervals. Next, we redefine our model to make sure we train from scratch again:

In [ ]:

model = GPT2LMHeadModel(config)

We can then define our optimizer, using the function from before to split the params for weight decay:

In [ ]:
from torch.optim import AdamW

optimizer = AdamW(get_grouped_params(model), lr=5e-4)

Now let's prepare the model, optimizer, and dataloaders so we can start training:

In [ ]:
from accelerate import Accelerator

accelerator = Accelerator(fp16=True)

model, optimizer, train_dataloader, eval_dataloader = Accelerator.prepare(
    model, optimizer, train_dataloader, eval_dataloader
)

Now that we have sent our train_dataloader to accelerator.prepare(), we can use its length to compute the number of training steps. Remember that we should always do this after preparing the dataloader, as that method will change its length. We use a classic linear schedule from the learning rate to 0:

In [ ]:
from transformers import get_scheduler

epochs=1
update_steps_per_epoch = len(train_dataloader)
training_steps = epochs * update_steps_per_epoch

lr_scheduler = get_scheduler(
    name='linear',
    optimizer=optimizer,
    num_warmup_steps=1_000,
    num_training_steps=training_steps,
)

Lastly, to push our model to the Hub, we will need to create a Repository object in a working folder

In [ ]:
from huggingface_hub import Repository, get_full_repo_name

model_name = 'codeparrot-ds-accelerate'
repo_name = get_full_repo_name(model_name)
repo_name

Then we can clone that repository in a local folder. If it already exists, this local folder should be an existing clone of the repository we are working with:

In [ ]:

output_dir = 'codeparrot-ds-accelerate'
repo = Repository(output_dir, clone_from=repo_name)

We can now upload anything we save in output_dir by calling the repo.push_to_hub() method. This will help us upload the intermediate models at the end of each epoch.

Before we train, let’s run a quick test to see if the evaluation function works properly:

In [ ]:
evaluate()

Those are very high values for loss and perplexity, but that's not surprising as we haven't trained the model yet. With that, we have everything prepared to write the core part of the training script: the training loop. In the training loop we iterate over the dataloader and pass the batches to the model. With the logits, we can then evaluate our custom loss function.

We scale the loss by the number of gradient accumulation steps so as not to creat larger losses when aggregating more steps. Before we optimize, we also clip the gradients for better convergence. Finally, every few steps we evaluate the model on the evaluation set with our new evaluate() function:

In [ ]:
from tqdm.notebook import tqdm

gradient_accumulation_steps = 8
eval_steps = 5_000

model.train()
completed_steps = 0
for epoch in range(num_train_epochs):
    for step, batch in tqdm(
        enumerate(train_dataloader, start=1), total=num_training_steps
    ):
        logits = model(batch["input_ids"]).logits
        loss = keytoken_weighted_loss(batch["input_ids"], logits, keytoken_ids)
        if step % 100 == 0:
            accelerator.print(
                {
                    "samples": step * samples_per_step,
                    "steps": completed_steps,
                    "loss/train": loss.item() * gradient_accumulation_steps,
                }
            )
        loss = loss / gradient_accumulation_steps
        accelerator.backward(loss)
        if step % gradient_accumulation_steps == 0:
            accelerator.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()
            completed_steps += 1
        if (step % (eval_steps * gradient_accumulation_steps)) == 0:
            eval_loss, perplexity = evaluate()
            accelerator.print({"loss/eval": eval_loss, "perplexity": perplexity})
            model.train()
            accelerator.wait_for_everyone()
            unwrapped_model = accelerator.unwrap_model(model)
            unwrapped_model.save_pretrained(output_dir, save_function=accelerator.save)
            if accelerator.is_main_process:
                tokenizer.save_pretrained(output_dir)
                repo.push_to_hub(
                    commit_message=f"Training in progress step {step}", blocking=False
                )

And that's it, we now have our own custom training loop for causal language models such as GPT-2 that we can further customize to our specific needs !!